In [2]:
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
from mne import read_labels_from_annot, read_source_spaces, setup_source_space
from mne.viz import plot_source_estimates
from mne.datasets import fetch_fsaverage
from nilearn import plotting
from nilearn.plotting import plot_surf_stat_map
import mne
import os

# Step 1: Load the .mat files containing the Mann-Whitney U test results
gc_in_time_data = scipy.io.loadmat('/Users/borjan/code/healthy-sleep-project/code/results/new_results_GC/gc_in_time_mw_effect_sizes.mat')
gc_out_time_data = scipy.io.loadmat('/Users/borjan/code/healthy-sleep-project/code/results/new_results_GC/gc_out_time_mw_effect_sizes.mat')
gc_in_freq_data = scipy.io.loadmat('/Users/borjan/code/healthy-sleep-project/code/results/new_results_GC/gc_in_freq_mw_effect_sizes.mat')
gc_out_freq_data = scipy.io.loadmat('/Users/borjan/code/healthy-sleep-project/code/results/new_results_GC/gc_out_freq_mw_effect_sizes.mat')


In [3]:
gc_in_time_data

{'__header__': b'MATLAB 5.0 MAT-file, Platform: MACA64, Created on: Thu Oct 31 12:16:35 2024',
 '__version__': '1.0',
 '__globals__': [],
 'mw_in_time_effect_sizes': array([[(array([[(array([[0.37931034]]), array([[20]], dtype=uint8), array([[0.70445742]])),
                 (array([[0.17241379]]), array([[17]], dtype=uint8), array([[0.86311223]])),
                 (array([[0.37931034]]), array([[20]], dtype=uint8), array([[0.70445742]])),
                 (array([[0.72413793]]), array([[25]], dtype=uint8), array([[0.46898106]])),
                 (array([[0.44827586]]), array([[21]], dtype=uint8), array([[0.65395412]])),
                 (array([[0.37931034]]), array([[20]], dtype=uint8), array([[0.70445742]])),
                 (array([[0.37931034]]), array([[20]], dtype=uint8), array([[0.70445742]])),
                 (array([[0.37931034]]), array([[20]], dtype=uint8), array([[0.70445742]])),
                 (array([[0.72413793]]), array([[25]], dtype=uint8), array([[0.46898106]])

In [4]:
# Corrected step to load custom labels
label_file_path = '/Users/borjan/code/healthy-sleep-project/code/results/labels.mat'
labels_data = scipy.io.loadmat(label_file_path)

# Extract the labels from the 'labels' field
# Since it's an array of strings, we need to handle the extraction accordingly
custom_labels = [label.strip() for label in labels_data['labels'].flatten()]

In [5]:
subjects_dir = '/Users/borjan/mne_data/MNE-fsaverage-data'
labels = mne.read_labels_from_annot('fsaverage', 'aparc', subjects_dir=subjects_dir)


Reading labels from parcellation...
   read 35 labels from /Users/borjan/mne_data/MNE-fsaverage-data/fsaverage/label/lh.aparc.annot
   read 34 labels from /Users/borjan/mne_data/MNE-fsaverage-data/fsaverage/label/rh.aparc.annot


In [6]:
import numpy as np
import matplotlib.pyplot as plt
from nilearn.plotting import plot_surf_stat_map

# Step 1: Match custom labels from .mat file with the loaded MNE Desikan-Killiany atlas labels

matched_labels = []
label_mapping = []  # Store indices of matched labels

# Iterate over your custom labels
for i, custom_label in enumerate(custom_labels):
    for mne_label in labels:  # Iterate over the MNE DK atlas labels
        if custom_label == mne_label.name:  # Compare the custom label to the MNE label name
            matched_labels.append(custom_label)
            label_mapping.append(i)
            break  # Stop searching once we find a match
    else:
        print(f"Warning: Custom label {custom_label} does not match any MNE labels and will be excluded.")


In [7]:
from mne.viz import Brain


# Initialize the brain visualization
subjects_dir = mne.datasets.sample.data_path() / "subjects"
labels_combined = mne.read_labels_from_annot('fsaverage', 'aparc', 'both', subjects_dir=subjects_dir)

brain = Brain('fsaverage', 'both', surf='pial', subjects_dir=subjects_dir,
              cortex='low_contrast', background='white', size=(800, 600))


Reading labels from parcellation...
   read 35 labels from /Users/borjan/mne_data/MNE-sample-data/subjects/fsaverage/label/lh.aparc.annot
   read 34 labels from /Users/borjan/mne_data/MNE-sample-data/subjects/fsaverage/label/rh.aparc.annot
Using pyvistaqt 3d backend.



2024-10-31 13:16:47.568 python[17616:20475934] +[IMKClient subclass]: chose IMKClient_Legacy
2024-10-31 13:16:47.568 python[17616:20475934] +[IMKInputSession subclass]: chose IMKInputSession_Legacy


In [8]:
# Function to add colored labels based on z-scores
def add_colored_labels(brain, labels, scores, cmap, norm):
    for label, score in zip(labels, scores):
        color = cmap(norm(score))
        brain.add_label(label, color=color, alpha=0.95, borders=False)


In [9]:
from pathlib import Path

# Define paths
output_dir = Path('/Users/borjan/code/healthy-sleep-project/code/results/new_results_GC/figures/')
output_dir.mkdir(parents=True, exist_ok=True)


# Clear previous labels
brain.remove_labels()

# Add the z-scores to the brain visualization
add_colored_labels(brain, labels_combined, zscores, cmap, norm)

# Show the brain from the top view
brain.show_view('dorsal')

# Save the figure
output_file_path = output_dir / 'W_v_N1_brain_viz.png'
brain.save_image(str(output_file_path))

brain.close()

# Display the plot
plt.show()

NameError: name 'zscores' is not defined

In [ ]:
# Step 5: Define function to extract the 'z' effect size for each region, condition, and frequency band
def extract_effect_sizes(effect_data, condition, band, label_mapping):
    effect_sizes = np.zeros(len(label_mapping))
    
    # Extract the 'z' values (effect sizes) for each region based on matched labels
    for idx, region_idx in enumerate(label_mapping):
        try:
            effect_sizes[idx] = effect_data['mw_in_effect_sizes'][condition][band][region_idx]['z'][0][0]
        except KeyError:
            effect_sizes[idx] = 0  # In case a region or value is missing

    return effect_sizes

# Step 6: Define plotting function for effect sizes
def plot_gc_effects(effect_sizes, condition_pair, band=None, title=None):
    stat_map = np.zeros(len(label_mapping))  # Only plot matched regions

    # Assign the effect size to the corresponding region
    for idx, region in enumerate(matched_labels):
        region_idx = mne_label_names.index(region)  # Find index of the matched label in MNE
        stat_map[region_idx] = effect_sizes[idx]  # Map the effect size to the brain region

    # Plot the stat map on the brain using nilearn's surface plotting tools
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    plot_surf_stat_map(fsaverage['infl_left'], stat_map, hemi='left', figure=fig, colorbar=True, cmap='coolwarm', 
                       threshold=0.1, title=title or f'{condition_pair} - {band if band else "Time-domain"}')
    
    # Plot right hemisphere (optional)
    plot_surf_stat_map(fsaverage['infl_right'], stat_map, hemi='right', figure=fig, colorbar=True, cmap='coolwarm', 
                       threshold=0.1)

# Step 7: Generate plots for the different conditions and frequency bands

# Time-domain plots (GC Inbound/Outbound)
for condition in ['N1', 'N2', 'N3', 'R_']:
    # Plot for inbound time-domain GC
    inbound_effect_sizes = extract_effect_sizes(gc_in_time_data, condition, 'time_domain', label_mapping)
    plot_gc_effects(inbound_effect_sizes, condition, title='GC Inbound - Time-domain')
    
    # Plot for outbound time-domain GC
    outbound_effect_sizes = extract_effect_sizes(gc_out_time_data, condition, 'time_domain', label_mapping)
    plot_gc_effects(outbound_effect_sizes, condition, title='GC Outbound - Time-domain')

# Frequency-domain plots (GC Inbound)
for condition in ['N1', 'N2', 'N3', 'R_']:
    for band in ['delta', 'theta', 'alpha', 'beta', 'gamma']:
        # Plot for inbound frequency-domain GC
        inbound_effect_sizes = extract_effect_sizes(gc_in_freq_data, condition, band, label_mapping)
        plot_gc_effects(inbound_effect_sizes, condition, band, title='GC Inbound - Frequency-domain')

# Frequency-domain plots (GC Outbound)
for condition in ['N1', 'N2', 'N3', 'R_']:
    for band in ['delta', 'theta', 'alpha', 'beta', 'gamma']:
        # Plot for outbound frequency-domain GC
        outbound_effect_sizes = extract_effect_sizes(gc_out_freq_data, condition, band, label_mapping)
        plot_gc_effects(outbound_effect_sizes, condition, band, title='GC Outbound - Frequency-domain')

plt.show()